## 2. 概述、线性代数和 NDArray

### 2.1 理论计算题

**已知向量** a=[2,-1,3]^T, b=[1,4,-2]^T，矩阵
A = [[2,1],[1,0],[-1,3]], B = [[0,2],[3,1]]

**1. 向量点积 a·b：**
a·b = 2(1) + (-1)(4) + 3(-2) = **-8**

**2. 矩阵乘法 A×B：**
A×B = [[3,5],[0,2],[9,1]]，形状为 **3×2**

**3. 向量 a 的 Frobenius 范数：**
‖a‖_F = √(4+1+9) = **√14**

### 2.2 编程题

使用 NumPy 完成以下任务：

In [ ]:
import numpy as np

# 1. 创建 3x4 的随机矩阵 X
np.random.seed(42)
X = np.random.randn(3, 4)
print("X (3x4):")
print(X)

# 2. 创建 4x2 的全 1 矩阵 Y
Y = np.ones((4, 2))
print("\nY (4x2):")
print(Y)

# 3. 计算矩阵乘法 Z = X @ Y
Z = X @ Y
print("\nZ = X @ Y (3x2):")
print(Z)

# 4. Z[0,1] 和 Z 的第 2 行
print(f"\nZ[0,1] = {Z[0, 1]}")
print(f"Z 的第 2 行: {Z[1, :]}")

# 5. Frobenius 范数
fro_norm = np.linalg.norm(Z, 'fro')
print(f"\nZ 的 Frobenius 范数: {fro_norm}")

## 3. 概率与统计

### 3.1 理论计算题（贝叶斯公式）

已知：
- P(患病) = 0.1% = 0.001
- P(阳性|患病) = 99% = 0.99（灵敏度）
- P(阳性|未患病) = 2% = 0.02（假阳性率）

求：P(患病|阳性)

P(患病|阳性) = 0.99 × 0.001 / (0.99 × 0.001 + 0.02 × 0.999) = 0.00099 / 0.02097 ≈ **4.72%**

### 3.2 编程题（中心极限定理模拟）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# 生成 n=10000 个 U(0,1)，重复 m=1000 次
n = 10000
m = 1000
means = np.array([np.random.uniform(0, 1, n).mean() for _ in range(m)])

# 绘制直方图 + 理论正态分布 PDF
mu = 0.5
var = 1.0 / 12
std_theory = np.sqrt(var / n)

x = np.linspace(means.min(), means.max(), 200)
pdf = stats.norm.pdf(x, mu, std_theory)

plt.figure(figsize=(10, 6))
plt.hist(means, bins=50, density=True, alpha=0.7, color='steelblue',
         edgecolor='white', label='Simulated means')
plt.plot(x, pdf, 'r-', linewidth=2,
         label=f'Theory N({mu}, {std_theory:.6f})')
plt.xlabel('Sample Mean')
plt.ylabel('Density')
plt.title('CLT Simulation (n=10000, m=1000)')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('CLT_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

actual_var = np.var(means, ddof=1)
print(f'Theoretical var: {std_theory**2:.8f}')
print(f'Actual var:      {actual_var:.8f}')
print(f'Theoretical std: {std_theory:.6f}')
print(f'Actual std:      {np.sqrt(actual_var):.6f}')

## 4. 导数、反向传播和复杂度

### 4.1 理论计算题

已知 z = (w1·x1 + w2·x2 - y)^2，其中 x1=2, x2=1, y=3。

dz/dw1 = 2(w1·x1 + w2·x2 - y) · x1
dz/dw2 = 2(w1·x1 + w2·x2 - y) · x2

当 w1=0.5, w2=1 时：inner = 1+1-3 = -1
dz/dw1 = -4, dz/dw2 = -2

### 4.2 编程题（手动反向传播）

In [ ]:
import numpy as np

x, w1, w2 = 2.0, 1.5, 0.5

# Forward
a = x * w1
b = a + w2
L = b ** 2
print(f'Forward: a={a}, b={b}, L={L}')

# Backward
dL_db = 2 * b
dL_dw1 = dL_db * x
dL_dw2 = dL_db
print(f'\nManual: dL/dw1={dL_dw1}, dL/dw2={dL_dw2}')

# Finite diff
eps = 1e-7
def fwd(a, b): return (x*a+b)**2
a1 = (fwd(w1+eps,w2)-fwd(w1-eps,w2))/(2*eps)
a2 = (fwd(w1,w2+eps)-fwd(w1,w2-eps))/(2*eps)
print(f'\nFinite diff: dL/dw1={a1:.6f}, dL/dw2={a2:.6f}')
print(f'Match: {abs(dL_dw1-a1):.2e}, {abs(dL_dw2-a2):.2e}')

## 5. 线性方法、基础优化和 Softmax 回归

### 5.1 理论计算题

L = (1/n) Σ(yi - (wxi + b))^2
dL/dw = -(2/n) Σ xi(yi - wxi - b)
dL/db = -(2/n) Σ (yi - wxi - b)

### 5.2 编程题（从零实现 Softmax 回归）

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.random.seed(42)
digits = load_digits()
X = digits.data / 16.0
y = digits.target
nc = 10
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

oh = np.zeros((y_tr.size, nc)); oh[np.arange(y_tr.size), y_tr] = 1

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def ce(p, t): return -np.sum(t*np.log(p+1e-8))/t.shape[0]

W = np.random.randn(X_tr.shape[1], nc)*0.01
b = np.zeros(nc)
lr, bs, ep = 0.1, 32, 50
n = X_tr.shape[0]

for e in range(ep):
    idx = np.random.permutation(n)
    Xs, ys = X_tr[idx], oh[idx]
    losses = []
    for i in range(0, n, bs):
        xb, yb = Xs[i:i+bs], ys[i:i+bs]
        p = softmax(xb@W+b)
        losses.append(ce(p, yb))
        g = (p-yb)/xb.shape[0]
        W -= lr*(xb.T@g)
        b -= lr*g.sum(axis=0)
    if (e+1)%10==0:
        print(f'Epoch {e+1:2d}/{ep}, Loss={np.mean(losses):.4f}')

acc = (softmax(X_te@W+b).argmax(1)==y_te).mean()
print(f'\nTest accuracy: {acc*100:.2f}%')

## 6. 最大似然估计和逻辑回归

### 6.1 理论计算题

已知 x_i ~ N(mu, sigma^2) i.i.d.

1. L(mu, sigma^2) = (2pi sigma^2)^(-n/2) exp(-1/(2sigma^2) sum((xi-mu)^2))
2. MLE mu: mu_hat = (1/n) sum(xi)
3. MLE sigma^2: sigma^2_hat = (1/n) sum((xi-mu_hat)^2)

### 6.2 编程题（二分类逻辑回归）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

np.random.seed(42)
X, y = make_blobs(n_samples=400, centers=2, n_features=2,
                   random_state=42, cluster_std=1.5)
X_tr, y_tr = X[:300], y[:300].reshape(-1,1)
X_te, y_te = X[300:], y[300:].reshape(-1,1)
print(f'Train: {X_tr.shape[0]}, Test: {X_te.shape[0]}')

def sig(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
def bce(p,t): return -np.mean(t*np.log(p+1e-8)+(1-t)*np.log(1-p+1e-8))

w = np.random.randn(2,1)*0.01
b = np.zeros((1,))
losses = []
for _ in range(1000):
    p = sig(X_tr@w+b)
    losses.append(bce(p,y_tr))
    d = (p-y_tr)/X_tr.shape[0]
    w -= 0.1*(X_tr.T@d)
    b = b - 0.1*np.sum(d, axis=0, keepdims=True)
print(f'Loss: {losses[-1]:.4f}')

fig, ax = plt.subplots(1,2,figsize=(14,5))
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-1,X[:,0].max()+1,200),
                      np.linspace(X[:,1].min()-1,X[:,1].max()+1,200))
pr = sig(np.c_[xx.ravel(),yy.ravel()]@w+b).reshape(xx.shape)
ax[0].contourf(xx,yy,pr,levels=50,alpha=0.6,cmap='RdYlBu')
ax[0].contour(xx,yy,pr,levels=[0.5],colors='black',linewidths=2)
ax[0].scatter(X_tr[y_tr.ravel()==0,0],X_tr[y_tr.ravel()==0,1],c='red',label='C0',alpha=0.6)
ax[0].scatter(X_tr[y_tr.ravel()==1,0],X_tr[y_tr.ravel()==1,1],c='blue',label='C1',alpha=0.6)
ax[0].set_title('Decision Boundary'); ax[0].legend()
ax[1].plot(losses,color='steelblue'); ax[1].set_title('Loss'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('logistic_regression.png',dpi=150,bbox_inches='tight'); plt.show()
acc = ((sig(X_te@w+b)>=0.5).astype(int)==y_te).mean()
print(f'Test accuracy: {acc*100:.2f}%')